In [1]:
#import mysql.connector
#from mysql.connector import Error
ConnectString = "mysql -h database-1.clqxqhhe6wft.us-east-1.rds.amazonaws.com -P 3306 -u admin -p'<Enter_DB_Password>' --ssl-verify-server-cert  --ssl-ca=/certs/global-bundle.pem mysql"

host=ConnectString[9:60]
user='admin'
password='Data608-Project'
database='CHESSBOT'


In [2]:

import warnings

# Suppress the specific Google Generative AI warning
warnings.filterwarnings("ignore")

try:
    import nextmove as nm
    import metrics as mt
    import movelist as ml
    import boardImage as bI
    import htmlPagetemplate as ht
    import nextmove as nm
    import snark as sn
    import re
    import mysql.connector
    from mysql.connector import Error
    from datetime import datetime
    

    
except Exception as e:
    # Catches any other unexpected exceptions
    print(f"An unexpected error occurred: {e}")


In [3]:
#this chunk is to define all values and variables that are passed back to the program from the webpage

SourceSpaceColoum = "d"  #from a - h
SourceSpaceRow = "2"  # from 1 - 8
DestinationSpaceColoum = "d" #from a - h
DestinationSpaceRow = "4"  # from 1 - 8


#AllMovesString = "1. e2e4 g7g6 2. d2d4 d7d6 3. g1f3 c7c6 4. h2h3 g8f6 5. c1g5 f6e4 6. d1e2 c8f5 7. b1d2 d8a5 8. c2c3 e4d2 9. g5d2 b8d7 10. b2b4 a5a3 11. f3g5 h7h5 12. e2c4 d6d5 13. c4e2 a3b2 14. e2d1 f5c2 15. d1c1 b2c1 16. a1c1 c2a4 17. f1d3 d7b6 18. e1g1 b6c4 19. d3c4 d5c4 20. d2f4 f8h6 21. f1e1 e8g8 22. e1e7 a8e8 23. e7b7 f7f6 24. g5e6 e8e6 25. f4h6 f8f7 26. b7b8 g8h7 27. h6f4 g6g5 28. f4d2 e6e2 29. d2e1 f7e7 30. g1f1 a4c2 31. b8c8 c2d3 32. c8c6 e2c2 33. f1g1 c2c1 34. c6f6 h5h4 35. g2g4 e7e1 36. g1g2 d3e4"
AllMovesString = "1. e2e4 g7g6"

# string built for current came

BotDifficulty = "1000"  #1000-Easy ,  1500 - Medium,  2500 - Hard,  5000 - Grandmaster
MetricDisplay = "1"   #1 - display metrics   0 - do not display
SnarkLevel = "evil" #off, neutral, positive, evil
RemarkFreq = "1"  #0 - off, 10 - low,  4 - Medium 1 - High

RemarkType = "general move"  #1) first move  2) general move  3)Black won 4) UniqueMovePosition 5)Illegal Move
GameCondition = "start" #start - default initial move ,  middle - state N as moves come and go ,    Wend - game has ended and no more moves possible (White won),  Bend - game has ended and no more moves possible (Black won),  Uend - game end, unique postions


In [4]:
#call NextMove


UserMove = SourceSpaceColoum +  SourceSpaceRow +  DestinationSpaceColoum + DestinationSpaceRow

#connect to database
connection = mysql.connector.connect(
    host=host,
    user=user,
    password=password,
    database='CHESSBOT' # Optional: leave out to create DB first
   )

if connection.is_connected():
    cursor = connection.cursor()
else:
    exit()

BotMove,Metrics,BoardLayout,GameCondition = nm.nextmove(BoardLayout,AllMovesString,UserMove,cursor,BotDifficulty,GameCondition)

print("GameCondition: ",GameCondition)

# append AllMovesString with new pair of moves
move_numbers = re.findall(r'(\d+)\.', AllMovesString)
next_num = int(move_numbers[-1]) + 1 if move_numbers else 1
AllMovesString += f" {next_num}. {UserMove} {BotMove}"


NameError: name 'BoardLayout' is not defined

In [ ]:
#call metrics
BoardLayout = mt.metrics(Metrics, BotMove, BoardLayout)

In [ ]:
#call boardimage ~ 64 times
for row in range(8, 0, -1):  # Corrected: start at 8, stop at 1, step -1
    FirstPartOfPage += '<tr><td>' + str(row) + "</td>"
    for column in ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h']:
        space_address = f"{column}{row}" 

        # Call your boardImage function
        PiecePictureName = bI.boardImage(BoardLayout, space_address)
        
        # Debug print
#        print(f"{space_address} -> {PiecePictureName}")

        FirstPartOfPage += f'<td><img src="{PiecePictureName}" width="100" alt="Chess Square {space_address}"></td>'
    FirstPartOfPage += '</tr>'

FirstPartOfPage += '<tr><td></td><td>A</td><td>B</td><td>C</td><td>D</td><td>E</td><td>F</td><td>G</td><td>H</td></tr>'


In [ ]:
#added Metric option  or display Metrics
MetricDecision = ""
if False:#MetricDisplay == "0" and len(AllMovesString) == 0:
    #added questionaire for metrics
    MetricDecision = '<tr><td colspan="2" style="border: 2px solid #555;">Display Metrics:<br><BIG><input type="radio" id="on" name="MetricDisplay" value="1" checked><label for="on">On</label>'
    MetricDecision += '<input type="radio" id="off" name="MetricDisplay" value="0"><label for="off">Off</label></BIG>'
    MetricDecision += '</td><td style="border: 2px solid #555;">Bot Difficulty: <br><select id="BotDiff" name="BotDifficulty" style="font-size: 34px; padding-left: 20px; padding-right: 20px;">'
    MetricDecision += '<option value="1000">Easy</option><option value="1500" selected>Medium</option><option value="2500">Hard</option><option value="5000">Grand Master</option></select>'
    
    MetricDecision += '</td><td style="border: 2px solid #555;">Bot Response Style:<br><select id="Snark" name="SnarkLevel" style="font-size: 34px; padding-left: 20px; padding-right: 20px;">'
    MetricDecision += '<option value="off">Off</option><option value="neutral" selected>Neutral</option><option value="positive">Encouraging</option><option value="evil">Snob</option></select>'

    MetricDecision += '</td><td colspan="2" style="border: 2px solid #555;">Bot Response Frequency:<br><select id="Freq" name="RemarkFreq" style="font-size: 34px; padding-left: 20px; padding-right: 20px;">'
    MetricDecision += '<option value="10">Low</option><option value="4" selected>Medium</option><option value="1">High</option></select>'
    
    MetricDecision += '</td></tr>'

elif MetricDisplay == "1":
    #display metrics
    MetricDecision = f'<tr><td colspan="2" style="border: 2px solid #555;">Number of games considered: {Metrics[0]}</td>'
    MetricDecision += '<td colspan="2" style="border: 2px solid #555;">Bot Moves considered:<br>'
    for row in Metrics[1]:
        MetricDecision += f'Move: {row[0]} >> {row[1]*100:.2f}%<br>'
    
    MetricDecision += '</td><td colspan="2" style="border: 2px solid #555;">Next move for user:<br>'
    
    for row in Metrics[2]:
        MetricDecision += f'Move: {row[0]} # of games: {row[1]}<br>'
    
    MetricDecision += '</td></tr>'

FourthPartOfPage = MetricDecision + FourthPartOfPage


In [ ]:
#output webpage raw
print(FirstPartOfPage)
print(SecondPartOfPage)
print(ThirdPartOfPage)
print(FourthPartOfPage)
print(FifthPartOfPage)